# LLM06 Excessive Agency — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM06 — Excessive Agency | **Risk Severity**: Critical

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM06 excessive agency test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

In [ ]:
%pip install okareo python-dotenv --quiet

In [1]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...x9vEA)
Category directory: /Users/mason/git/compliance-owasp/owasp/LLM06-excessive-agency


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [2]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM06-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM06-autonomous-action from autonomous-action.jsonl
  ✓ Registered: LLM06-autonomous-action (ID: 8610a2c8-1478-4b2f-be6c-839dfba2f066)
Uploading scenario: LLM06-permission-escalation from permission-escalation.jsonl
  ✓ Registered: LLM06-permission-escalation (ID: 01c83e3b-f029-4cae-9503-75dfc1407b31)
Uploading scenario: LLM06-unauthorized-tool-invocation from unauthorized-tool-invocation.jsonl
  ✓ Registered: LLM06-unauthorized-tool-invocation (ID: 4a6a92d2-a2d3-4bdc-a28f-f99eda500842)

Total scenarios uploaded: 3


### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [8]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


Registering check: LLM06-agency-boundary-drift-detector from agency-boundary-drift-detector.md
  ✓ Registered: LLM06-agency-boundary-drift-detector (ID: e5362fc4-3808-4ac7-b1a0-4d6b45118948)
Registering check: LLM06-excessive-agency-detector from excessive-agency-detector.md
  ✓ Registered: LLM06-excessive-agency-detector (ID: f72a343c-39fc-4df4-835c-0646a0edbfed)

Total checks registered: 2


### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

In [4]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver data dict

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_artifact(md_path, default_temperature=0.6)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


Registering driver: LLM06-approval-gate-bypasser from approval-gate-bypasser.md
  ✓ Registered: LLM06-approval-gate-bypasser (ID: b583a36b-4a70-4629-86bf-0fc291e6e110)
Registering driver: LLM06-privilege-escalator from privilege-escalator.md
  ✓ Registered: LLM06-privilege-escalator (ID: e40c0dcf-2d1a-47c1-ae8f-ce1ffb8d231c)
Registering driver: LLM06-tool-scope-manipulator from tool-scope-manipulator.md
  ✓ Registered: LLM06-tool-scope-manipulator (ID: ca1b81be-fa83-442a-8fbb-78dba3e9fb86)

Total drivers registered: 3


### Artifact Upload Summary

In [5]:
print("=" * 60)
print("LLM06 Excessive Agency — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

LLM06 Excessive Agency — Artifact Upload Summary

Scenarios (3):
  • LLM06-autonomous-action → 8610a2c8-1478-4b2f-be6c-839dfba2f066
  • LLM06-permission-escalation → 01c83e3b-f029-4cae-9503-75dfc1407b31
  • LLM06-unauthorized-tool-invocation → 4a6a92d2-a2d3-4bdc-a28f-f99eda500842

Checks (2):
  • LLM06-agency-boundary-drift-detector → e5362fc4-3808-4ac7-b1a0-4d6b45118948
  • LLM06-excessive-agency-detector → f72a343c-39fc-4df4-835c-0646a0edbfed

Drivers (3):
  • LLM06-approval-gate-bypasser → b583a36b-4a70-4629-86bf-0fc291e6e110
  • LLM06-privilege-escalator → e40c0dcf-2d1a-47c1-ae8f-ce1ffb8d231c
  • LLM06-tool-scope-manipulator → ca1b81be-fa83-442a-8fbb-78dba3e9fb86

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.json` file (copy `owasp/target.json.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

The target is registered as a `CustomEndpointTarget` with `TurnConfig` defining how to send messages.
All LLM06 scenarios use `okareo.run_simulation()` with `max_turns=10, first_turn="target"` and dedicated
adversarial driver personas. Each scenario is paired with its own driver via the `SCENARIO_DRIVER_MAP`.

In [ ]:
# Target loaded from owasp/target.json. To use a different config: target = build_target(CATEGORY_DIR, config_path="target.prod.json")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

EXCESSIVE_AGENCY_CHECK = "LLM06-excessive-agency-detector"
DRIFT_DETECTOR_CHECK   = "LLM06-agency-boundary-drift-detector"
CHECK_NAMES = [EXCESSIVE_AGENCY_CHECK, DRIFT_DETECTOR_CHECK]
MAX_TURNS = 1

SCENARIO_DRIVER_MAP = {
    "LLM06-unauthorized-tool-invocation": "LLM06-tool-scope-manipulator",
    "LLM06-permission-escalation":        "LLM06-privilege-escalator",
    "LLM06-autonomous-action":            "LLM06-approval-gate-bypasser",
}

✓ Target agent: my-agent


### Multi-Turn Simulations — All 3 LLM06 Scenarios

Each scenario runs via `okareo.run_simulation()` with `max_turns=10, first_turn="target"`.
Each scenario is paired with its dedicated adversarial driver persona via `SCENARIO_DRIVER_MAP`.
Both the excessive agency detector and agency boundary drift detector checks are applied to every run.

In [10]:
simulation_results = {}  # scenario_name -> test run result

for scenario_name, driver_name in SCENARIO_DRIVER_MAP.items():
    if scenario_name not in registered_scenarios:
        print(f"⚠ Scenario {scenario_name} not found in registered scenarios — skipping")
        continue

    if driver_name not in registered_drivers:
        print(f"⚠ Driver {driver_name} not found in registered drivers — skipping {scenario_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Running simulation: {scenario_name}")
    print(f"Driver: {driver_name} | Max turns: {MAX_TURNS}")
    print(f"{'='*60}")

    try:
        driver_reg = registered_drivers[driver_name]
        multi_turn_driver = Driver(
            temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.7,
            name=driver_name,
            prompt_template=driver_reg.prompt_template,
        )

        scenario = registered_scenarios[scenario_name]

        simulation_run = okareo.run_simulation(
            target=target,
            driver=multi_turn_driver,
            name=f"LLM06 Simulation — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="target",
            scenario=scenario,
            max_turns=MAX_TURNS,
            checks=CHECK_NAMES,
        )
        simulation_results[scenario_name] = simulation_run
        print(f"  ✓ Simulation complete: {simulation_run.id}")
        if hasattr(simulation_run, "app_link") and simulation_run.app_link:
            print(f"  View: {simulation_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        simulation_results[scenario_name] = None


Running simulation: LLM06-unauthorized-tool-invocation
Driver: LLM06-tool-scope-manipulator | Max turns: 10
  ✓ Simulation complete: 7f87fa61-c938-4acb-8064-989d4174bcd6
  View: http://localhost:3000/project/acfafd81-c856-4aac-887a-d5ea89a87335/eval/7f87fa61-c938-4acb-8064-989d4174bcd6

Running simulation: LLM06-permission-escalation
Driver: LLM06-privilege-escalator | Max turns: 10


KeyboardInterrupt: 

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("LLM06 EXCESSIVE AGENCY — EVALUATION RESULTS")
print("OWASP Category: LLM06 | Risk Severity: Critical")
print("=" * 60)

print(f"\n{'Scenario':<46} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)
for name, result in simulation_results.items():
    if result is None:
        print(f"{name:<46} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<46} {'COMPLETE':<10} {link}")

errors = sum(1 for r in simulation_results.values() if r is None)
print(f"\nTotal evaluated: {len(simulation_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")

### Detailed Results (Optional)

Retrieve per-row scores and conversation transcripts for any completed simulation run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)